### Gravitational wave inference

This tutorial demonstrates how to run parameter estimation on a reduced parameter space for an injected gravitational wave signal.

This example estimates the masses using a uniform prior in both component masses and distance using a uniform in comoving volume prior on luminosity distance between luminosity distances of 100 Mpc and 5 Gpc, the cosmology is [Planck15](https://docs.astropy.org/en/stable/api/astropy.cosmology.realizations.Planck15.html).


I have just translated [this Python script](https://github.com/bilby-dev/bilby/blob/main/examples/gw_examples/injection_examples/fast_tutorial.py) into a Jupyter notebook. All credits go to the original authors. 

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

In [2]:
import bilby

In [3]:
#bilby.core.sampler.get_implemented_samplers()

In [4]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="bilby")
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

In [5]:
# These make plots look nicer
import matplotlib.pyplot as plt
font_value = 20

# Set global Matplotlib settings for a clean look without grids
plt.rcParams['axes.grid'] = False      # Disable grid lines for all axes
plt.rcParams['grid.color'] = 'none'    # Just in case, ensure grid color is invisible
# Set matplotlib to render LaTeX fonts
plt.rcParams.update({
    "mathtext.fontset" : "stix",
    "font.family" : "STIXGeneral",
    "axes.labelsize": font_value ,           # Font size for axis labels
    "axes.titlesize": font_value ,           # Font size for titles
    "font.size": font_value ,                # General font size
    "legend.fontsize": font_value ,          # Font size for legend
    "xtick.labelsize": font_value ,          # Font size for x-axis ticks
    "ytick.labelsize": font_value ,          # Font size for y-axis ticks
})

from matplotlib_inline.backend_inline import set_matplotlib_formats
# Enable retina display output in Jupyter
set_matplotlib_formats("retina")

In [6]:
# Set the duration and sampling frequency of the data segment that we're
# going to inject the signal into
duration = 4.0
sampling_frequency = 2048.0
minimum_frequency = 20

In [7]:
# Specify the output directory and the name of the simulation.
outdir = "outdir_gw"
label = "fast_tutorial"
bilby.core.utils.setup_logger(outdir=outdir, label=label)#, log_level="DEBUG")

In [8]:
# Set up a random seed for result reproducibility.  This is optional
bilby.core.utils.random.seed(88170235)

In [9]:
# We are going to inject a binary black hole waveform.  We first establish a
# dictionary of parameters that includes all of the different waveform
# parameters, including masses of the two black holes (mass_1, mass_2),
# spins of both black holes (a, tilt, phi), etc.
injection_parameters = dict(
    mass_1=36.0,
    mass_2=29.0,
    a_1=0.4,
    a_2=0.3,
    tilt_1=0.5,
    tilt_2=1.0,
    phi_12=1.7,
    phi_jl=0.3,
    luminosity_distance=2000.0,
    theta_jn=0.4,
    psi=2.659,
    phase=1.3,
    geocent_time=1126259642.413,
    ra=1.375,
    dec=-1.2108,
)

In [10]:
# evaluate chirp mass and mass ratio
print('chirp mass = ', bilby.gw.conversion.component_masses_to_chirp_mass(injection_parameters['mass_1'], injection_parameters['mass_2']), 'Msun')
print('mass ratio = ', bilby.gw.conversion.component_masses_to_mass_ratio(injection_parameters['mass_1'], injection_parameters['mass_2']))

chirp mass =  28.09555579546043 Msun
mass ratio =  0.8055555555555556


In [11]:
# Fixed arguments passed into the source model
waveform_arguments = dict(
    waveform_approximant="IMRPhenomPv2",
    reference_frequency=50.0,
    minimum_frequency=minimum_frequency,
)

# Create the waveform_generator using a LAL BinaryBlackHole source function
waveform_generator = bilby.gw.WaveformGenerator(
    duration=duration,
    sampling_frequency=sampling_frequency,
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
    parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters,
    waveform_arguments=waveform_arguments,
)

18:43 bilby INFO    : Waveform generator instantiated: WaveformGenerator(duration=4.0, sampling_frequency=2048.0, start_time=0, frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole, time_domain_source_model=None, parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters, waveform_arguments={'waveform_approximant': 'IMRPhenomPv2', 'reference_frequency': 50.0, 'minimum_frequency': 20})


In [12]:
# Set up interferometers.  In this case we'll use two interferometers
# (LIGO-Hanford (H1), LIGO-Livingston (L1). These default to their design
# sensitivity
ifos = bilby.gw.detector.InterferometerList(["H1", "L1"])
ifos.set_strain_data_from_power_spectral_densities(
    sampling_frequency=sampling_frequency,
    duration=duration,
    start_time=injection_parameters["geocent_time"] - 2,
)
ifos.inject_signal(
    waveform_generator=waveform_generator, parameters=injection_parameters
)

18:43 bilby INFO    : Injected signal in H1:
18:43 bilby INFO    :   optimal SNR = 11.78
18:43 bilby INFO    :   matched filter SNR = 10.70-1.01j
18:43 bilby INFO    :   mass_1 = 36.0
18:43 bilby INFO    :   mass_2 = 29.0
18:43 bilby INFO    :   a_1 = 0.4
18:43 bilby INFO    :   a_2 = 0.3
18:43 bilby INFO    :   tilt_1 = 0.5
18:43 bilby INFO    :   tilt_2 = 1.0
18:43 bilby INFO    :   phi_12 = 1.7
18:43 bilby INFO    :   phi_jl = 0.3
18:43 bilby INFO    :   luminosity_distance = 2000.0
18:43 bilby INFO    :   theta_jn = 0.4
18:43 bilby INFO    :   psi = 2.659
18:43 bilby INFO    :   phase = 1.3
18:43 bilby INFO    :   geocent_time = 1126259642.413
18:43 bilby INFO    :   ra = 1.375
18:43 bilby INFO    :   dec = -1.2108
18:43 bilby INFO    : Injected signal in L1:
18:43 bilby INFO    :   optimal SNR = 9.53
18:43 bilby INFO    :   matched filter SNR = 7.71+0.41j
18:43 bilby INFO    :   mass_1 = 36.0
18:43 bilby INFO    :   mass_2 = 29.0
18:43 bilby INFO    :   a_1 = 0.4
18:43 bilby INFO 

[{'plus': array([0.-0.j, 0.-0.j, 0.-0.j, ..., 0.-0.j, 0.-0.j, 0.-0.j], shape=(4097,)),
  'cross': array([0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j], shape=(4097,))},
 {'plus': array([0.-0.j, 0.-0.j, 0.-0.j, ..., 0.-0.j, 0.-0.j, 0.-0.j], shape=(4097,)),
  'cross': array([0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j], shape=(4097,))}]

In [13]:
# Set up a PriorDict, which inherits from dict.
# By default we will sample all terms in the signal models.  However, this will
# take a long time for the calculation, so for this example we will set almost
# all of the priors to be equall to their injected values.  This implies the
# prior is a delta function at the true, injected value.  In reality, the
# sampler implementation is smart enough to not sample any parameter that has
# a delta-function prior.
# The above list does *not* include mass_1, mass_2, theta_jn and luminosity
# distance, which means those are the parameters that will be included in the
# sampler.  If we do nothing, then the default priors get used.
priors = bilby.gw.prior.BBHPriorDict()
for key in [
    "a_1",
    "a_2",
    "tilt_1",
    "tilt_2",
    "phi_12",
    "phi_jl",
    "psi",
    "ra",
    "dec",
    "geocent_time",
    "phase",
    "theta_jn"
]:
    priors[key] = bilby.core.prior.DeltaFunction(injection_parameters[key])
    
priors['chirp_mass'] = bilby.gw.prior.UniformInComponentsChirpMass(minimum=15, maximum=50, name='chirp_mass', latex_label='$\\mathcal{M}$', unit=None, boundary='reflective')
priors['mass_ratio'] = bilby.gw.prior.UniformInComponentsMassRatio(minimum=0.5, maximum=1, name='mass_ratio', latex_label='$q$', unit=None, boundary='reflective', equal_mass=False)
priors['luminosity_distance'] = bilby.gw.prior.UniformSourceFrame(minimum=100.0, maximum=5000.0, cosmology='Planck15', name='luminosity_distance', latex_label='$D_\mathrm{L}$', unit='Mpc', boundary='reflective')

del priors['mass_1']
del priors['mass_2']

# Perform a check that the prior does not extend to a parameter space longer than the data
priors.validate_prior(duration, minimum_frequency)

18:43 bilby INFO    : No prior given, using default BBH priors in /Users/filippo/miniconda3/envs/bayes_env/lib/python3.11/site-packages/bilby/gw/prior_files/precessing_spins_bbh.prior.


True

In [14]:
priors

{'mass_ratio': bilby.gw.prior.UniformInComponentsMassRatio(minimum=0.5, maximum=1, name='mass_ratio', latex_label='$q$', unit=None, boundary='reflective', equal_mass=False),
 'chirp_mass': bilby.gw.prior.UniformInComponentsChirpMass(minimum=15, maximum=50, name='chirp_mass', latex_label='$\\mathcal{M}$', unit=None, boundary='reflective'),
 'luminosity_distance': bilby.gw.prior.UniformSourceFrame(minimum=100.0, maximum=5000.0, cosmology='Planck15', name='luminosity_distance', latex_label='$D_\\mathrm{L}$', unit='Mpc', boundary='reflective'),
 'dec': DeltaFunction(peak=-1.2108, name=None, latex_label=None, unit=None),
 'ra': DeltaFunction(peak=1.375, name=None, latex_label=None, unit=None),
 'theta_jn': DeltaFunction(peak=0.4, name=None, latex_label=None, unit=None),
 'psi': DeltaFunction(peak=2.659, name=None, latex_label=None, unit=None),
 'phase': DeltaFunction(peak=1.3, name=None, latex_label=None, unit=None),
 'a_1': DeltaFunction(peak=0.4, name=None, latex_label=None, unit=None),
 

In [15]:
# Initialise the likelihood by passing in the interferometer data (ifos) and
# the waveform generator
likelihood = bilby.gw.GravitationalWaveTransient(
    interferometers=ifos, waveform_generator=waveform_generator
)


In [ ]:
# Run sampler.  In this case we're going to use the `dynesty` sampler
result = bilby.run_sampler(
    likelihood=likelihood,
    priors=priors,
    
    #sampler="dynesty", # best sampler for GW inference, implementing nested sampling
    
    sampler="emcee",
    nsteps=10_000,
    nwalkers=500,
    nburn=500,
    
    npool=4, # base this parameter on the number of cores you have on you laptop, check with htop
    injection_parameters=injection_parameters,
    outdir=outdir,
    label=label,
    result_class=bilby.gw.result.CBCResult,
    clean=True,
)

18:43 bilby INFO    : Running for label 'fast_tutorial', output will be saved to 'outdir_gw'
18:43 bilby INFO    : Using lal version 7.7.0
18:43 bilby INFO    : Using lal git version Branch: None;Tag: lal-v7.7.0;Id: ef36dfdf49181b9b376a867b55a14463783de545;;Builder: Adam Mercer <adam.mercer@ligo.org>;Repository status: CLEAN: All modifications committed
18:43 bilby INFO    : Using lalsimulation version 6.2.0
18:43 bilby INFO    : Using lalsimulation git version Branch: None;Tag: lalsimulation-v6.2.0;Id: 1338470a6165fb4b4c98bccdd0efe961f05bc8e0;;Builder: Adam Mercer <adam.mercer@ligo.org>;Repository status: CLEAN: All modifications committed
18:43 bilby INFO    : Analysis priors:
18:43 bilby INFO    : mass_ratio=bilby.gw.prior.UniformInComponentsMassRatio(minimum=0.5, maximum=1, name='mass_ratio', latex_label='$q$', unit=None, boundary='reflective', equal_mass=False)
18:43 bilby INFO    : chirp_mass=bilby.gw.prior.UniformInComponentsChirpMass(minimum=15, maximum=50, name='chirp_mass', l

emcee: Exception while calling your likelihood function:
  params: [7.88684482e-01 2.81902900e+01 2.39223232e+03]
  args: []
  kwargs: {}
  exception:
emcee: Exception while calling your likelihood function:
  params: [9.73181651e-01 4.42026673e+01 4.94975671e+03]
  args: []
  kwargs: {}
  exception:
emcee: Exception while calling your likelihood function:
  params: [7.93960340e-01 2.81051951e+01 2.42651363e+03]
  args: []
  kwargs: {}
  exception:
emcee: Exception while calling your likelihood function:
  params: [7.67269589e-01 2.82825552e+01 2.42494979e+03]
  args: []
  kwargs: {}
  exception:


In [ ]:
# Make a corner plot.
result.plot_corner()